# Chapitre 7 : utiliser des exemples (few-shot prompting)

- [Cours](#cours)
- [Exercices](#exercices)
- [Espace de test](#espace-de-test)

## Configuration

Exécutez la cellule de configuration suivante pour charger votre clé API et définir la fonction utilitaire `get_completion`.

In [ ]:
!pip install anthropic

# Importer la bibliothèque standard de Python pour les expressions régulières
import re
import anthropic

# Récupérer les variables API_KEY et MODEL_NAME depuis le store IPython
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt="", prefill=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ]
    )
    return message.content[0].text

---

## Cours

**Donner à Claude des exemples de ce que vous attendez (ou de ce que vous ne voulez pas) est extrêmement efficace** pour :
- obtenir la bonne réponse ;
- obtenir la réponse dans le bon format.

Ce type de prompt s'appelle le « **few-shot prompting** ». Vous rencontrerez aussi les termes « zero-shot », « one-shot » ou « n-shot » : le nombre de « shots » désigne le nombre d'exemples fournis dans le prompt.

### Exemples

Imaginez que vous développez un « robot parent » qui répond aux questions des enfants. **Par défaut, la réponse de Claude est plutôt formelle et mécanique.** De quoi briser le cœur d'un enfant.

In [ ]:
# Prompt
PROMPT = "Will Santa bring me presents on Christmas?"

# Afficher la réponse de Claude
print(get_completion(PROMPT))

Vous pourriez prendre le temps de décrire le ton voulu, mais il est bien plus simple de **donner à Claude quelques exemples de réponses idéales**.

In [ ]:
# Prompt
PROMPT = """Please complete the conversation by writing the next line, speaking as "A".
Q: Is the tooth fairy real?
A: Of course, sweetie. Wrap up your tooth and put it under your pillow tonight. There might be something waiting for you in the morning.
Q: Will Santa bring me presents on Christmas?"""

# Afficher la réponse de Claude
print(get_completion(PROMPT))

Dans l'exemple de formatage suivant, nous pourrions guider Claude pas à pas avec des instructions pour extraire des noms et des professions puis les formater exactement comme nous le voulons. Nous pouvons aussi **lui fournir quelques exemples correctement formatés, à partir desquels il généralise**. Remarquez la balise `<individuals>` dans le tour `assistant`, qui met Claude sur la bonne voie dès le départ.

In [ ]:
# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = """Silvermist Hollow, a charming village, was home to an extraordinary group of individuals.
Among them was Dr. Liam Patel, a neurosurgeon who revolutionized surgical techniques at the regional medical center.
Olivia Chen was an innovative architect who transformed the village's landscape with her sustainable and breathtaking designs.
The local theater was graced by the enchanting symphonies of Ethan Kovacs, a professionally-trained musician and composer.
Isabella Torres, a self-taught chef with a passion for locally sourced ingredients, created a culinary sensation with her farm-to-table restaurant, which became a must-visit destination for food lovers.
These remarkable individuals, each with their distinct talents, contributed to the vibrant tapestry of life in Silvermist Hollow.
<individuals>
1. Dr. Liam Patel [NEUROSURGEON]
2. Olivia Chen [ARCHITECT]
3. Ethan Kovacs [MISICIAN AND COMPOSER]
4. Isabella Torres [CHEF]
</individuals>

At the heart of the town, Chef Oliver Hamilton has transformed the culinary scene with his farm-to-table restaurant, Green Plate. Oliver's dedication to sourcing local, organic ingredients has earned the establishment rave reviews from food critics and locals alike.
Just down the street, you'll find the Riverside Grove Library, where head librarian Elizabeth Chen has worked diligently to create a welcoming and inclusive space for all. Her efforts to expand the library's offerings and establish reading programs for children have had a significant impact on the town's literacy rates.
As you stroll through the charming town square, you'll be captivated by the beautiful murals adorning the walls. These masterpieces are the work of renowned artist, Isabella Torres, whose talent for capturing the essence of Riverside Grove has brought the town to life.
Riverside Grove's athletic achievements are also worth noting, thanks to former Olympic swimmer-turned-coach, Marcus Jenkins. Marcus has used his experience and passion to train the town's youth, leading the Riverside Grove Swim Team to several regional championships.
<individuals>
1. Oliver Hamilton [CHEF]
2. Elizabeth Chen [LIBRARIAN]
3. Isabella Torres [ARTIST]
4. Marcus Jenkins [COACH]
</individuals>

Oak Valley, a charming small town, is home to a remarkable trio of individuals whose skills and dedication have left a lasting impact on the community.
At the town's bustling farmer's market, you'll find Laura Simmons, a passionate organic farmer known for her delicious and sustainably grown produce. Her dedication to promoting healthy eating has inspired the town to embrace a more eco-conscious lifestyle.
In Oak Valley's community center, Kevin Alvarez, a skilled dance instructor, has brought the joy of movement to people of all ages. His inclusive dance classes have fostered a sense of unity and self-expression among residents, enriching the local arts scene.
Lastly, Rachel O'Connor, a tireless volunteer, dedicates her time to various charitable initiatives. Her commitment to improving the lives of others has been instrumental in creating a strong sense of community within Oak Valley.
Through their unique talents and unwavering dedication, Laura, Kevin, and Rachel have woven themselves into the fabric of Oak Valley, helping to create a vibrant and thriving small town."""

# Pré-remplissage de la réponse de Claude
PREFILL = "<individuals>"

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))

Si vous voulez expérimenter avec les prompts du cours sans modifier le contenu ci-dessus, descendez tout en bas du notebook jusqu'à l'[**Espace de test**](#espace-de-test).

---

## Exercices
- [Exercice 7.1 - Formater des e-mails par l'exemple](#exercice-71---formater-des-e-mails-par-lexemple)

### Exercice 7.1 - Formater des e-mails par l'exemple
Nous reprenons l'exercice 6.2, mais cette fois en modifiant le `PROMPT` pour y mettre des exemples « few-shot » d'e-mails avec leur classification (et leur format) corrects, afin que Claude donne la bonne réponse. La *dernière* lettre de la réponse de Claude doit être la lettre de la catégorie.

Reportez-vous aux commentaires à côté de chaque e-mail de la liste `EMAILS` si vous ne savez plus quelle lettre correspond à quel e-mail.

Rappel des catégories :
- (A) Pre-sale question (question avant achat)
- (B) Broken or defective item (article cassé ou défectueux)
- (C) Billing question (question de facturation)
- (D) Other (please explain) (autre, à expliquer)

In [ ]:
# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = """Please classify this email as either green or blue: {email}"""

# Pré-remplissage de la réponse de Claude
PREFILL = ""

# Contenu de la variable, stocké sous forme de liste
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Billing question
    "How did I get here I am not good with computer.  Halp." # (D) Other (please explain)
]

# Catégories correctes, stockées en liste de listes car un e-mail peut avoir plusieurs catégories correctes
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Parcourir la liste des e-mails
for i,email in enumerate(EMAILS):
    
    # Insérer le texte de l'e-mail dans la variable email
    formatted_prompt = PROMPT.format(email=email)
   
    # Obtenir la réponse de Claude
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Corriger la réponse de Claude
    grade = any([bool(re.search(ans, response[-1])) for ans in ANSWERS[i]])
    
    # Afficher la réponse de Claude
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")

❓ Pour obtenir un indice, exécutez la cellule ci-dessous.

In [ ]:
from hints import exercise_7_1_hint; print(exercise_7_1_hint)

Toujours bloqué ? Exécutez la cellule ci-dessous pour voir un exemple de solution.

In [ ]:
from hints import exercise_7_1_solution; print(exercise_7_1_solution)

### Bravo !

Si vous avez résolu tous les exercices jusqu'ici, vous pouvez passer au chapitre suivant. Bon prompting !

---

## Espace de test

Cet espace vous permet d'expérimenter librement avec les exemples de prompts de la leçon et de les modifier pour voir l'effet sur les réponses de Claude.

In [ ]:
# Prompt
PROMPT = "Will Santa bring me presents on Christmas?"

# Afficher la réponse de Claude
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = """Please complete the conversation by writing the next line, speaking as "A".
Q: Is the tooth fairy real?
A: Of course, sweetie. Wrap up your tooth and put it under your pillow tonight. There might be something waiting for you in the morning.
Q: Will Santa bring me presents on Christmas?"""

# Afficher la réponse de Claude
print(get_completion(PROMPT))

In [ ]:
# Modèle de prompt avec un emplacement pour le contenu de la variable
PROMPT = """Silvermist Hollow, a charming village, was home to an extraordinary group of individuals.
Among them was Dr. Liam Patel, a neurosurgeon who revolutionized surgical techniques at the regional medical center.
Olivia Chen was an innovative architect who transformed the village's landscape with her sustainable and breathtaking designs.
The local theater was graced by the enchanting symphonies of Ethan Kovacs, a professionally-trained musician and composer.
Isabella Torres, a self-taught chef with a passion for locally sourced ingredients, created a culinary sensation with her farm-to-table restaurant, which became a must-visit destination for food lovers.
These remarkable individuals, each with their distinct talents, contributed to the vibrant tapestry of life in Silvermist Hollow.
<individuals>
1. Dr. Liam Patel [NEUROSURGEON]
2. Olivia Chen [ARCHITECT]
3. Ethan Kovacs [MISICIAN AND COMPOSER]
4. Isabella Torres [CHEF]
</individuals>

At the heart of the town, Chef Oliver Hamilton has transformed the culinary scene with his farm-to-table restaurant, Green Plate. Oliver's dedication to sourcing local, organic ingredients has earned the establishment rave reviews from food critics and locals alike.
Just down the street, you'll find the Riverside Grove Library, where head librarian Elizabeth Chen has worked diligently to create a welcoming and inclusive space for all. Her efforts to expand the library's offerings and establish reading programs for children have had a significant impact on the town's literacy rates.
As you stroll through the charming town square, you'll be captivated by the beautiful murals adorning the walls. These masterpieces are the work of renowned artist, Isabella Torres, whose talent for capturing the essence of Riverside Grove has brought the town to life.
Riverside Grove's athletic achievements are also worth noting, thanks to former Olympic swimmer-turned-coach, Marcus Jenkins. Marcus has used his experience and passion to train the town's youth, leading the Riverside Grove Swim Team to several regional championships.
<individuals>
1. Oliver Hamilton [CHEF]
2. Elizabeth Chen [LIBRARIAN]
3. Isabella Torres [ARTIST]
4. Marcus Jenkins [COACH]
</individuals>

Oak Valley, a charming small town, is home to a remarkable trio of individuals whose skills and dedication have left a lasting impact on the community.
At the town's bustling farmer's market, you'll find Laura Simmons, a passionate organic farmer known for her delicious and sustainably grown produce. Her dedication to promoting healthy eating has inspired the town to embrace a more eco-conscious lifestyle.
In Oak Valley's community center, Kevin Alvarez, a skilled dance instructor, has brought the joy of movement to people of all ages. His inclusive dance classes have fostered a sense of unity and self-expression among residents, enriching the local arts scene.
Lastly, Rachel O'Connor, a tireless volunteer, dedicates her time to various charitable initiatives. Her commitment to improving the lives of others has been instrumental in creating a strong sense of community within Oak Valley.
Through their unique talents and unwavering dedication, Laura, Kevin, and Rachel have woven themselves into the fabric of Oak Valley, helping to create a vibrant and thriving small town."""

# Pré-remplissage de la réponse de Claude
PREFILL = "<individuals>"

# Afficher la réponse de Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print("USER TURN:")
print(PROMPT)
print("\nASSISTANT TURN:")
print(PREFILL)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT, prefill=PREFILL))